# Phase 2: Feature Engineering & Modeling
## 1. Import Libraries

In [27]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

## 2. Load Cleaned Data

In [28]:
df = pd.read_parquet('../datasets/cleaned_data.parquet')
print(f"Loaded shape: {df.shape}")

# Handle missing values and infs (sometimes they persist from phase 1)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
print(f"Shape after dropping NaNs/Infs: {df.shape}")

Loaded shape: (2522362, 71)
Shape after dropping NaNs/Infs: (2520798, 71)


## 3. Feature Selection & Engineering

In [29]:
# Encode Labels
le = LabelEncoder()
df['Label_Encoded'] = le.fit_transform(df['Label'])
joblib.dump(le, '../models/label_encoder.pkl')
print("Classes mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# Feature Selection (Keeping ~18 important features based on standard CIC-IDS2017 practices)
selected_features = [
    'Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 
    'Fwd Packet Length Mean', 'Bwd Packet Length Mean',
    'Flow Bytes/s', 'Flow Packets/s', 'Packet Length Mean', 'Packet Length Std', 
    'SYN Flag Count', 'ACK Flag Count', 'FIN Flag Count', 'RST Flag Count', 
    'PSH Flag Count', 'URG Flag Count'
]

# Ensure selected features exist in columns (ignoring case/spaces issues by matching strictly)
# If some names differ, adjust `selected_features` accordingly.
available_cols = df.columns.tolist()
features_to_use = [f for f in selected_features if f in available_cols]
print(f"Using {len(features_to_use)} features.")

X = df[features_to_use]
y = df['Label_Encoded']
joblib.dump(features_to_use, '../models/selected_features.pkl')

Classes mapping: {'BENIGN': np.int64(0), 'Bot': np.int64(1), 'DDoS': np.int64(2), 'DoS GoldenEye': np.int64(3), 'DoS Hulk': np.int64(4), 'DoS Slowhttptest': np.int64(5), 'DoS slowloris': np.int64(6), 'FTP-Patator': np.int64(7), 'Heartbleed': np.int64(8), 'Infiltration': np.int64(9), 'PortScan': np.int64(10), 'SSH-Patator': np.int64(11), 'Web Attack � Brute Force': np.int64(12), 'Web Attack � Sql Injection': np.int64(13), 'Web Attack � XSS': np.int64(14)}
Using 18 features.


['../models/selected_features.pkl']

## 4. Train-Test Split and Scaling

In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.pkl')
print("Data scaled and Scaler saved.")

Data scaled and Scaler saved.


## 5. Handle Class Imbalance

In [31]:
# Find BENIGN encoded value
benign_idx = le.transform(['BENIGN'])[0]

# SMOTE for minority, RandomUnderSampler for majority
# Using sampling_strategy='auto' or dictionary
# For large datasets, exact balancing can be too heavy, so we adjust dict manually if needed.
# Note: To avoid long runtimes, we limit the resampling

resampling_pipeline = Pipeline([
    ('under', RandomUnderSampler(sampling_strategy={benign_idx: 100000}, random_state=42)),
    ('smote', SMOTE(random_state=42))
])

print("Applying Resampling...")
X_resampled, y_resampled = resampling_pipeline.fit_resample(X_train_scaled, y_train)
print("Resampling Done.")

Applying Resampling...
Resampling Done.


## 6. Model Training & Evaluation

In [32]:
# Initialize Models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Naive Bayes': GaussianNB(),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    # 'SVM': SVC() # SVM can be extremely slow on large datasets
}

best_model_name = 'Random Forest'
best_model = None

for name, model in models.items():
    print(f"\n--- Training {name} ---")
    model.fit(X_resampled, y_resampled)
    y_pred = model.predict(X_test_scaled)
    
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    
    if name == best_model_name:
        best_model = model

joblib.dump(best_model, '../models/random_forest.pkl')
print(f"\nBest model ({best_model_name}) saved to models/random_forest.pkl")


--- Training Logistic Regression ---
                            precision    recall  f1-score   support

                    BENIGN       1.00      0.69      0.82    419012
                       Bot       0.01      0.99      0.02       390
                      DDoS       0.69      0.99      0.81     25603
             DoS GoldenEye       0.39      0.90      0.55      2057
                  DoS Hulk       0.85      0.94      0.89     34569
          DoS Slowhttptest       0.10      0.88      0.17      1046
             DoS slowloris       0.18      0.60      0.27      1077
               FTP-Patator       0.18      1.00      0.30      1186
                Heartbleed       0.10      1.00      0.17         2
              Infiltration       0.00      1.00      0.00         7
                  PortScan       0.66      0.97      0.79     18139
               SSH-Patator       0.14      0.91      0.24       644
  Web Attack � Brute Force       0.00      0.06      0.00       294
Web Attac